# Resize and Concatenate

This notebook resamples the WorldView-3 SWIR mosaic to match the spatial resolution and grid of the VNIR mosaic, then combines the two datasets into a single multispectral image.

The output is a 16-band GeoTIFF consisting of the 8 VNIR bands followed by the 8 SWIR bands, preserving the georeferencing information of the VNIR mosaic.

### Input

- VNIR mosaic (`VNIR_mosaic.tif`)
- SWIR mosaic (`SWIR_mosaic.tif`)

### Output

- Resampled SWIR mosaic (`SWIR_upsampled.tif`)
- Combined 16-band mosaic (`VNIR_SWIR_stack.tif`)

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling


# Input and output directories

data_dir = Path("../output")

vnir_path = data_dir / "VNIR_mosaic.tif"
swir_path = data_dir / "SWIR_mosaic.tif"

swir_up_path = data_dir / "SWIR_upsampled.tif"
output_path = data_dir / "VNIR_SWIR_stack.tif"


# Resample the SWIR mosaic to match the VNIR spatial resolution

with rasterio.open(vnir_path) as vnir, rasterio.open(swir_path) as swir:

# Set up a profile for the new upscaled SWIR
    profile_up = swir.profile.copy()
    profile_up.update({
        'crs': vnir.crs,
        'transform': vnir.transform,
        'width': vnir.width,
        'height': vnir.height
    })

# Create a SWIR file upscaled to 1 m using georeferenced resampling
    with rasterio.open(swir_up_path, 'w', **profile_up) as dst:
        for i in range(1, swir.count + 1):
            reproject(
                source=rasterio.band(swir, i),
                destination=rasterio.band(dst, i),
                src_transform=swir.transform,
                src_crs=swir.crs,
                dst_transform=vnir.transform,
                dst_crs=vnir.crs,
                resampling=Resampling.cubic 
            )

print(f"SWIR mosaic resampled and saved as:\n{swir_up_path.name}")


# Stack VNIR and SWIR bands into a single 16-band image

# Read and concatenate the two mosaics
with rasterio.open(vnir_path) as vnir, rasterio.open(swir_up_path) as swir_up:

    # Consistency check
    if (vnir.height, vnir.width) != (swir_up.height, swir_up.width):
        raise ValueError("Spatial dimensions do not match after resampling.")
    if vnir.crs != swir_up.crs or vnir.transform != swir_up.transform:
        raise ValueError("CRS and transform do not match: check the alignment.")

    # Read the data
    vnir_data = vnir.read()       # (8, h, w)
    swir_data = swir_up.read()    # (8, h, w)

    # Band stacking
    stacked = np.concatenate([vnir_data, swir_data], axis=0)  # (16, h, w)

    # Final profile
    profile_out = vnir.profile.copy()
    profile_out.update(count=stacked.shape[0])

    # Saving
    with rasterio.open(output_path, 'w', **profile_out) as dst:
        dst.write(stacked)

print(f"Combined VNIR-SWIR stack saved as:\n{output_path.name}")
